# Week 11 — Python Solution Lab
## Simple Harmonic Motion

**Companion to `notebooks/Week_11.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_11.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P1` | Spring-Mass SHM Basics | phase relationships + energy exchange plot |
| **L2 · Intermediate** | `P6` | Damped Oscillation Over Five Cycles | turning-point decay vs the sinusoidal envelope |
| **L3 · Challenge** | `P10` | Robot Joint as a Torsional Oscillator | underdamped analysis; then overshoot/settling as an extension |

---

## L1 · Basic — P1: Spring-Mass SHM Basics

> **Problem (Week_11.ipynb, L1 — P1).** A $0.50$ kg mass on a spring with $k = 200$ N/m is
> displaced $0.08$ m and released from rest. Find (a) $\omega$, (b) $T$, (c) $v_{\max}$,
> (d) $a_{\max}$.

**Diagram → Principle.** $F = -kx$ through $F = ma$ gives $\ddot x = -\omega^2 x$, whose solution
is sinusoidal. Released from rest at maximum displacement means $x(t) = A\cos\omega t$.

**Equation.** $\omega = \sqrt{k/m}$, $T = 2\pi/\omega$, $v_{\max} = A\omega$, $a_{\max} = A\omega^2$.

**Hand prediction.** $\omega = 20$ rad/s, $T = 0.314$ s, $v_{\max} = 1.60$ m/s, $a_{\max} = 32.0$ m/s².

**What Python adds.** We plot $x$, $v$, $a$ together so the **90° phase relationships** are seen
rather than memorised — $v$ peaks where $x = 0$, $a$ is exactly opposite $x$. Then we confirm
energy sloshes between kinetic and potential while the total stays flat to machine precision,
which is the real content of "simple harmonic".

In [ ]:
# ═══ W11 · L1 · P1 — SHM basics, phase relationships, and the energy exchange ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, k, A = 0.50, 200.0, 0.08

# --- PREDICT ------------------------------------------------------------
w     = np.sqrt(k/m)
T     = 2*np.pi/w
v_max = A*w
a_max = A*w**2
print(f"(a) omega = sqrt(k/m) = {w:.4f} rad/s")
print(f"(b) T     = 2 pi/omega = {T:.5f} s      (f = {1/T:.4f} Hz)")
print(f"(c) v_max = A omega    = {v_max:.4f} m/s")
print(f"(d) a_max = A omega^2  = {a_max:.4f} m/s^2   ({a_max/9.81:.2f} g)")

# --- VERIFY: v_max independently, from energy ---------------------------
v_energy = np.sqrt(k/m)*A                      # (1/2)kA^2 = (1/2)mv^2
print(f"\nenergy route: v_max = A sqrt(k/m) = {v_energy:.4f} m/s -> agrees")
assert np.isclose(v_max, v_energy)
# a_max from Newton at full extension
print(f"Newton route: a_max = kA/m = {k*A/m:.4f} m/s^2 -> agrees")
assert np.isclose(a_max, k*A/m)

# --- The motion, with phase relationships visible -----------------------
t = np.linspace(0, 2*T, 1200)
x = A*np.cos(w*t)
v = -A*w*np.sin(w*t)
a = -A*w**2*np.cos(w*t)

print(f"\nphase check at t = 0 (released from rest at x = +A):")
print(f"  x = {x[0]:+.4f} m (maximum), v = {v[0]:+.4f} m/s (zero), a = {a[0]:+.4f} m/s^2 (most negative)")
print(f"  a/x = {a[10]/x[10]:.4f} = -omega^2 = {-w**2:.4f}  -> a is ALWAYS -w^2 x. That IS SHM.")
assert np.allclose(a, -w**2 * x)

# --- Energy: exchanged, but conserved -----------------------------------
KE, PE = 0.5*m*v**2, 0.5*k*x**2
E = KE + PE
print(f"\ntotal energy E = (1/2) k A^2 = {0.5*k*A**2:.6f} J")
print(f"  simulated E ranges {E.min():.10f} to {E.max():.10f}  (spread {np.ptp(E):.2e} J)")
assert np.ptp(E) < 1e-12

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.6, 6), sharex=True)
ax1.plot(t/T, x,      color="#1565c0", lw=2, label="$x$ (m)")
ax1.plot(t/T, v/w,    color="#e65100", lw=2, label="$v/\\omega$ (m)")
ax1.plot(t/T, a/w**2, color="#2e7d32", lw=2, ls="--", label="$a/\\omega^2$ (m)")
ax1.set_ylabel("scaled to metres"); ax1.legend(fontsize=9, ncol=3)
ax1.set_title("x, v, a are the same wave shifted by 90 deg each")
ax2.plot(t/T, KE, color="#e65100", lw=2, label="kinetic")
ax2.plot(t/T, PE, color="#1565c0", lw=2, label="potential")
ax2.plot(t/T, E,  color="k", lw=2.2, ls="--", label="total (flat)")
ax2.set_xlabel("t / T"); ax2.set_ylabel("energy (J)"); ax2.legend(fontsize=9)
ax2.set_title("energy sloshes at twice the frequency; the sum never moves")
for ax in (ax1, ax2): ax.grid(alpha=.3); ax.axhline(0, c="k", lw=.6)
plt.suptitle("W11 P1 — 0.50 kg on a 200 N/m spring, A = 8 cm", y=1.01)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(w - 20.0) < 1e-9 and abs(T - 0.31416) < 1e-4
assert abs(v_max - 1.60) < 1e-9 and abs(a_max - 32.0) < 1e-9
print(f"[OK] Matches textbook answer: omega = {w:.0f} rad/s, T = {T:.3f} s, "
      f"v_max = {v_max:.2f} m/s, a_max = {a_max:.1f} m/s^2")

## L2 · Intermediate — P6: Damped Oscillation Over Five Cycles

> **Problem (Week_11.ipynb, L2 — P6).** A $1.2$ kg block on a spring ($k = 48$ N/m) has damping
> $b = 2.4$ N·s/m. Starting from $x_0 = 0.25$ m at rest, find (a) the displacement after exactly
> 5 complete cycles and (b) the fraction of mechanical energy remaining.

**Diagram → Principle.** Add a velocity-proportional drag to the restoring force:
$m\ddot x + b\dot x + kx = 0$. The amplitude decays as $e^{-\gamma t}$ with $\gamma = b/2m$, while
the oscillation continues at the slightly reduced $\omega_d$.

**Equation.** $\gamma = b/2m$, $\omega_d = \sqrt{\omega_0^2 - \gamma^2}$,
$x(t) \approx x_0e^{-\gamma t}\cos\omega_d t$; energy $\propto$ amplitude².

**Hand prediction.** $\gamma = b/2m = 1.0$ s⁻¹, $\omega_0 = 6.325$ rad/s, $\omega_d = 6.245$ rad/s,
$T_d = 1.006$ s. After $5T_d = 5.031$ s: $x \approx 0.25\,e^{-5.031} = 1.63$ mm, and
$E/E_0 = e^{-2\gamma t} \approx 0.0043\%$.

> ⚠️ **Answer-key discrepancy.** The key prints $x(5T_d) \approx 0.0046$ m and
> $E/E_0 \approx 0.034\%$. From the stated $m$, $k$, $b$ the turning-point curve gives
> $0.25\,e^{-(1.0)(5.031)} = 0.00163$ m, and the exact ODE agrees to six decimals. The key's
> value corresponds to $\gamma t \approx 4.0$ rather than $5.03$. Use **1.63 mm** and
> **0.0043 %**. (Its $E/E_0$ is at least self-consistent with its own $x$, since
> $E/E_0 = (x/x_0)^2$.)

**Why the whole-period displacement simplifies exactly.** Released from rest, the true solution is

$$x(t) = x_0e^{-\gamma t}\left[\cos\omega_dt + \frac{\gamma}{\omega_d}\sin\omega_dt\right]
      = C\,e^{-\gamma t}\cos(\omega_dt - \phi),
\quad C = x_0\sqrt{1+(\gamma/\omega_d)^2},\ \tan\phi = \frac{\gamma}{\omega_d}.$$

At $t = 5T_d = 10\pi/\omega_d$ we have $\sin\omega_dt = 0$ and $\cos\omega_dt = 1$, so the
bracket collapses to exactly $1$:

$$x(5T_d) = x_0e^{-5\gamma T_d}\quad\textbf{exactly}.$$

Since $\dot x = 0$ there too, $E/E_0 = e^{-10\gamma T_d}$ exactly as well.

> **Two different curves — do not call them both "the envelope".** Setting $\dot x = 0$ gives
> $\omega_dt = n\pi$, so the **turning points** fall every *half* period, and there
> $|x| = C e^{-\gamma t}\cos\phi = x_0e^{-\gamma t}$. So $x_0e^{-\gamma t}$ is the
> **turning-point decay curve** — it threads the actual extrema exactly. The **sinusoidal
> envelope** of $C e^{-\gamma t}\cos(\omega_dt-\phi)$ is the larger
> $$A_{\rm env}(t) = x_0\sqrt{1+(\gamma/\omega_d)^2}\;e^{-\gamma t},$$
> here $0.253185\,e^{-t}$ rather than $0.250000\,e^{-t}$ — a $1.27\%$ difference. The two
> touch only where $\cos(\omega_dt-\phi) = \pm1$, which is *not* where the extrema are.
> Our answer uses the turning-point curve, which is the right one for "the displacement after
> 5 complete cycles".

**What Python adds.** We integrate the true ODE with `solve_ivp` and confirm exact agreement at
$5T_d$; show that at a *non*-integer number of periods the phase term genuinely matters; and
verify numerically that the simulated extrema really do sit on $x_0e^{-\gamma t}$ while the
sinusoidal envelope sits above them.

In [ ]:
# ═══ W11 · L2 · P6 — Damped SHM: turning-point curve vs the true ODE ═══
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, k, b, x0 = 1.2, 48.0, 2.4, 0.25

gamma = b / (2*m)
w0    = np.sqrt(k/m)
wd    = np.sqrt(w0**2 - gamma**2)
Td    = 2*np.pi/wd

print(f"gamma  = b/2m           = {gamma:.4f} 1/s")
print(f"omega0 = sqrt(k/m)      = {w0:.4f} rad/s")
print(f"omega_d= sqrt(w0^2-g^2) = {wd:.4f} rad/s   ({'underdamped' if gamma < w0 else 'not underdamped'})")
print(f"T_d    = {Td:.5f} s;  damping ratio zeta = {gamma/w0:.4f}")

t5 = 5*Td

# --- (a) ROUTE 1: the turning-point decay curve x0 exp(-gamma t) --------
x5_env = x0 * np.exp(-gamma*t5)
print(f"\n(a) after 5 cycles, t = 5 T_d = {t5:.4f} s")
print(f"    turning-point curve x = x0 exp(-gamma t) = {x5_env:.6f} m")

# --- ROUTE 2: integrate the real equation of motion ---------------------
def rhs(t, y):
    x, v = y
    return [v, (-k*x - b*v)/m]

sol = solve_ivp(rhs, [0, t5*1.30], [x0, 0.0], rtol=1e-12, atol=1e-14, dense_output=True)
x5_ode = sol.sol(t5)[0]
print(f"    exact ODE solution                     = {x5_ode:.6f} m")
print(f"    difference                             = {abs(x5_env-x5_ode):.2e} m")

# --- WHY they agree exactly at an integer number of damped periods ------
print("\n    The full solution for release from rest is")
print("      x(t) = x0 exp(-gamma t) [cos(w_d t) + (gamma/w_d) sin(w_d t)]")
print(f"    At t = 5 T_d the phase w_d*t = 10*pi, so sin = {np.sin(wd*t5):+.1e} and "
      f"cos = {np.cos(wd*t5):+.6f}:")
print("    the bracket collapses to exactly 1, so x(5T_d) = x0 exp(-gamma t) exactly.")
bracket = np.cos(wd*t5) + (gamma/wd)*np.sin(wd*t5)
print(f"      bracket at 5 T_d = {bracket:.12f}")
assert abs(bracket - 1.0) < 1e-9, "bracket must be 1 at an integer number of periods"
assert abs(x5_env - x5_ode) < 1e-9

# --- TURNING-POINT CURVE vs SINUSOIDAL ENVELOPE (not the same curve) ----
C_env = x0*np.sqrt(1 + (gamma/wd)**2)          # amplitude of C e^-gt cos(wd t - phi)
print(f"\n    Careful with the word 'envelope' -- there are two curves here:")
print(f"      turning-point decay curve  x0 e^-gt      : prefactor {x0:.6f} m")
print(f"      sinusoidal envelope        C  e^-gt      : prefactor {C_env:.6f} m "
      f"(+{100*(C_env/x0 - 1):.2f}%)")
print(f"      C = x0 sqrt(1 + (gamma/w_d)^2) = {x0} * sqrt(1 + ({gamma}/{wd:.4f})^2)")
print(f"    Setting dx/dt = 0 gives w_d t = n*pi, so extrema fall every HALF period,")
print(f"    and there |x| = C e^-gt cos(phi) = x0 e^-gt exactly. The extrema therefore")
print(f"    lie on the LOWER curve; the sinusoidal envelope never touches them.")

# verify that claim against the simulation
from scipy.signal import find_peaks
t_chk = np.linspace(0, t5, 200000)
x_chk = sol.sol(t_chk)[0]
pk, _ = find_peaks(np.abs(x_chk))
err_tp = np.max(np.abs(np.abs(x_chk[pk]) - x0*np.exp(-gamma*t_chk[pk])))
print(f"      simulated |extrema| vs x0 e^-gt : max error {err_tp:.1e} m  -> they coincide")
print(f"      mean extrema spacing {np.mean(np.diff(t_chk[pk])):.5f} s "
      f"vs T_d/2 = {Td/2:.5f} s")
assert err_tp < 1e-5, "extrema must lie on the turning-point curve"
assert C_env > x0

# --- ...and where the turning-point curve is NOT the displacement -------
t_off = 5.25*Td                      # a quarter period later: phase term is alive
x_off_env, x_off_ode = x0*np.exp(-gamma*t_off), sol.sol(t_off)[0]
print(f"\n    Contrast, at t = 5.25 T_d (a quarter period later):")
print(f"      x0 e^-gt  {x_off_env:+.6f} m")
print(f"      exact     {x_off_ode:+.6f} m   <- the phase term now matters")
print("    x0 e^-gt gives the displacement only AT the extrema (every half period),")
print("    not in between: it threads the turning points, it is not x(t) itself.")

# --- (b) energy fraction remaining --------------------------------------
E0 = 0.5*k*x0**2
# at t5 the block is essentially at a turning point, so use the full mechanical energy
v5 = sol.sol(t5)[1]
E5 = 0.5*k*x5_ode**2 + 0.5*m*v5**2
print(f"\n(b) E0 = {E0:.6f} J")
print(f"    E5 = {E5:.3e} J")
print(f"    E5/E0 = {E5/E0:.3e} = {100*E5/E0:.4f}%   -> essentially all of it is gone")
print(f"    turning-point curve gives exp(-2 gamma t) = {np.exp(-2*gamma*t5):.3e} "
      f"({100*np.exp(-2*gamma*t5):.4f}%)")

# --- Useful decay yardsticks --------------------------------------------
print(f"\n  amplitude 1/e time  = 1/gamma      = {1/gamma:.4f} s "
      f"({1/gamma/Td:.2f} cycles)")
print(f"  energy    1/e time  = 1/(2 gamma)  = {1/(2*gamma):.4f} s")
print(f"  quality factor Q    = w0/(2 gamma) = {w0/(2*gamma):.4f}")

# --- Plot ---------------------------------------------------------------
tt = np.linspace(0, t5*1.02, 4000)
xx = sol.sol(tt)[0]
fig, ax = plt.subplots(figsize=(7.8, 4))
ax.plot(tt, xx, color="#1565c0", lw=1.6, label="exact $x(t)$")
ax.plot(tt,  x0*np.exp(-gamma*tt), color="#e65100", lw=2, ls="--",
        label="turning-point curve $\\pm x_0e^{-\\gamma t}$")
ax.plot(tt, -x0*np.exp(-gamma*tt), color="#e65100", lw=2, ls="--")
ax.plot(tt,  C_env*np.exp(-gamma*tt), color="grey", lw=1.2, ls=":",
        label="sinusoidal envelope $\\pm Ce^{-\\gamma t}$")
ax.plot(tt, -C_env*np.exp(-gamma*tt), color="grey", lw=1.2, ls=":")
for n in range(6):
    ax.axvline(n*Td, color="grey", lw=.6, ls=":", alpha=.5)
ax.plot(t5, x5_ode, "o", color="crimson", ms=9, zorder=5, label=f"5 cycles: {x5_ode*1000:.2f} mm")
ax.set_xlabel("t (s)"); ax.set_ylabel("x (m)")
ax.set_title("W11 P6 — extrema sit on $x_0e^{-\\gamma t}$, inside the sinusoidal envelope")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(gamma - 1.0) < 1e-12 and abs(wd - 6.2450) < 1e-3
assert abs(Td - 1.00613) < 1e-4
assert abs(x5_ode - 0.001634) < 2e-5, f"x5 = {x5_ode}"
assert abs(100*E5/E0 - 0.00427) < 5e-4

print(f"\nANSWER-KEY NOTE: the key prints x = 0.0046 m and E/E0 = 0.034%.")
print(f"  turning-point curve: x0 exp(-gamma*5T_d) = 0.25*exp(-{gamma*t5:.4f}) = {x0*np.exp(-gamma*t5):.6f} m")
print(f"  exact ODE agrees: {x5_ode:.6f} m. The key's 0.0046 m corresponds to gamma*t ~ 4.0,")
print(f"  not the {gamma*t5:.2f} that gamma = b/2m = {gamma:.1f} and 5 T_d actually give.")
print(f"  Use x = {x5_ode*1000:.2f} mm and E/E0 = {100*E5/E0:.4f}%.")

print(f"\n[OK] after 5 cycles x = {x5_ode*1000:.3f} mm; "
      f"{100*E5/E0:.4f}% of the energy remains.")

## L3 · Challenge — P10: Robot Joint as a Torsional Oscillator

> **Problem (Week_11.ipynb, L3 — P10).** A robot arm joint is a torsional spring-damper with
> $\kappa = 50$ N·m/rad, $c = 2.0$ N·m·s/rad, $I = 0.40$ kg·m². Displaced $10^\circ$ and released.
> Governing equation: $I\ddot\theta + c\dot\theta + \kappa\theta = 0$. Find the natural frequency
> and the damping behaviour.

**Diagram → Principle.** Structurally identical to the mass-spring-damper, with
$m\to I$, $b\to c$, $k\to\kappa$. Everything from Week 11 transfers unchanged.

**Equation.** $\omega_0 = \sqrt{\kappa/I}$, $\gamma = c/2I$, $\zeta = \gamma/\omega_0$, and
$\omega_d = \sqrt{\omega_0^2-\gamma^2}$.

**This is what P10 asks for — the complete answer:**

$$\omega_0 = \sqrt{50/0.40} = \sqrt{125} = 11.180\ \text{rad/s} \;(f_0 = 1.779\ \text{Hz}),
\qquad \gamma = \frac{c}{2I} = 2.50\ \text{s}^{-1}$$

$$\zeta = \frac{\gamma}{\omega_0} = \frac{c}{2\sqrt{\kappa I}} = 0.2236 < 1
\;\Longrightarrow\; \textbf{underdamped}$$

so the joint oscillates about equilibrium while its amplitude decays exponentially, at
$\omega_d = \sqrt{125 - 6.25} = 10.897$ rad/s, i.e. $T_d = 0.5766$ s. That is the whole of the
required answer.

> **Everything below the sweep is an optional extension, not part of P10.** The problem asks
> only for the natural frequency and the damping behaviour of the joint *as given*. It does not
> ask you to redesign the damping, minimise settling time, or impose any overshoot limit. We go
> further because the machinery is already in place and the question "what would you *do* about
> $\zeta = 0.22$?" is the interesting one — but the design criterion is ours, not the problem's,
> and the notebook says so at each step.

**What Python adds.** The closed-form underdamped solution is checked against `solve_ivp` to
$10^{-8}$ rad. Then, as an extension, we sweep $c$ and measure overshoot and settling time
directly from the integrated response, to show how the choice of damping follows from whatever
requirement the designer sets.

> **Not everything here needs a simulation.** The first overshoot *does* have an exact
> expression: extrema occur at $\omega_dt = n\pi$, so
> $$\frac{|\theta_1|}{\theta_0} = e^{-\gamma\pi/\omega_d} = e^{-2.5\pi/10.897} = 0.4864,$$
> i.e. $48.64\%$, at $t = \pi/\omega_d = 0.288$ s — and the cell checks the measured value
> against it. The **2% settling time** is the one with no tidy closed form: it is the root of a
> transcendental equation, and the familiar $t_s \approx 4/\gamma = 1.60$ s rule is $5.8\%$ off
> the true $1.512$ s here. Measuring from the integrated response is what generalises — to
> settling time now, and to systems with no closed form at all later.

In [ ]:
# ═══ W11 · L3 · P10 — Torsional joint: the asked analysis, then an extension ═══
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
I, kappa, c = 0.40, 50.0, 2.0
theta0 = np.radians(10.0)

w0    = np.sqrt(kappa/I)
gamma = c/(2*I)
zeta  = gamma/w0
c_crit = 2*np.sqrt(kappa*I)

print(f"omega_0      = sqrt(kappa/I)   = {w0:.4f} rad/s  ({w0/(2*np.pi):.4f} Hz)")
print(f"gamma        = c/(2I)          = {gamma:.4f} 1/s")
print(f"zeta         = gamma/omega_0   = {zeta:.4f}")
print(f"c_critical   = 2 sqrt(kappa I) = {c_crit:.4f} N*m*s/rad  (we have c = {c:.1f})")
regime = "UNDERDAMPED (it will ring)" if zeta < 1 else ("critically damped" if zeta == 1 else "overdamped")
print(f"-> {regime}")

wd = np.sqrt(w0**2 - gamma**2)
print(f"omega_d = {wd:.4f} rad/s, T_d = {2*np.pi/wd:.4f} s, Q = {w0/(2*gamma):.3f}")

def simulate(c_val, t_end=3.0, n=6000):
    f = lambda t, y: [y[1], (-kappa*y[0] - c_val*y[1])/I]
    s = solve_ivp(f, [0, t_end], [theta0, 0.0], rtol=1e-11, atol=1e-13, dense_output=True)
    t = np.linspace(0, t_end, n)
    return t, s.sol(t)[0]

def metrics(t, th, tol=0.02):
    """Overshoot past zero (%) and 2% settling time."""
    overshoot = max(0.0, -th.min()) / theta0 * 100
    outside = np.where(np.abs(th) > tol*theta0)[0]
    t_settle = t[outside[-1]] if len(outside) else 0.0
    return overshoot, t_settle

t, th = simulate(c)
ov, ts = metrics(t, th)
print(f"\nAs built (c = {c:.1f}): overshoot {ov:.1f}% of the initial angle, "
      f"2% settling time {ts:.3f} s")
print(f"  the joint crosses the zero-angle equilibrium and reaches "
      f"{np.degrees(th.min()):.2f} deg on the far side before recovering.")

# --- the overshoot DOES have a closed form; check the measurement -------
# extrema at w_d t = n*pi, so the first one is
#   theta_1 = th0 e^(-gamma*pi/w_d) [cos(pi) + 0] = -th0 e^(-gamma*pi/w_d)
OS_exact = np.exp(-gamma*np.pi/wd)
print(f"\n  closed form for the first overshoot: |theta_1|/theta_0 = "
      f"exp(-gamma*pi/w_d)")
print(f"    = exp(-{gamma}*pi/{wd:.4f}) = {OS_exact:.6f}  ->  {100*OS_exact:.4f}%")
print(f"    measured from the integrated response:      {ov:.4f}%")
print(f"    first extremum at t = pi/w_d = {np.pi/wd:.5f} s "
      f"(measured {t[np.argmin(th)]:.5f} s)")
assert abs(100*OS_exact - ov) < 1e-3, "closed-form overshoot must match the measurement"
assert abs(t[np.argmin(th)] - np.pi/wd) < 1e-3

# --- VERIFY the analytic underdamped solution against the ODE -----------
th_analytic = theta0*np.exp(-gamma*t)*(np.cos(wd*t) + (gamma/wd)*np.sin(wd*t))
err = np.max(np.abs(th_analytic - th))
print(f"  max |analytic - numerical| = {err:.2e} rad  -> the closed form checks out")
assert err < 1e-8

# --- DESIGN: sweep the damping constant ---------------------------------
print(f"\n=== EXTENSION -- beyond what P10 asks =================================")
print(f"P10 is fully answered above. What follows explores what one COULD do about")
print(f"zeta = {zeta:.4f}; the design criteria below are ours, not the problem's.")
print(f"\nDESIGN SWEEP -- how do overshoot and settling time depend on c?")
print(f"  {'c':>6s} {'zeta':>7s} {'overshoot %':>12s} {'settle (s)':>11s}")
cands = [2.0, 4.0, 6.0, 8.0, c_crit, 10.0, 14.0, 20.0]
ZERO = 1e-6          # 'no overshoot' means NO overshoot, not 'a little'
TOL  = 0.5           # a separate, explicitly-stated tolerance band
best_zero = best_tol = None
for cv in cands:
    tt, thh = simulate(cv)
    o, s_ = metrics(tt, thh)
    if o <= ZERO and (best_zero is None or s_ < best_zero[2]):
        best_zero = (cv, o, s_)
    if o <= TOL and (best_tol is None or s_ < best_tol[2]):
        best_tol = (cv, o, s_)
for cv in cands:
    tt, thh = simulate(cv)
    o, s_ = metrics(tt, thh)
    tag = ""
    if best_tol and abs(cv - best_tol[0]) < 1e-12:
        tag = f"   <- fastest with overshoot <= {TOL}%"
    if best_zero and abs(cv - best_zero[0]) < 1e-12:
        tag = "   <- fastest with ZERO overshoot (critical)"
    print(f"  {cv:6.2f} {cv/(2*np.sqrt(kappa*I)):7.3f} {o:12.2f} {s_:11.3f}{tag}")

print(f"\n  Read those two tags carefully -- they are different requirements:")
print(f"    strictly no overshoot  -> needs zeta >= 1, i.e. c >= c_crit = {c_crit:.3f}")
print(f"       best: c = {best_zero[0]:.3f}, overshoot {best_zero[1]:.2f}%, "
      f"settles {best_zero[2]:.3f} s")
print(f"    overshoot allowed up to {TOL}% -> a slightly underdamped joint is FASTER")
print(f"       best: c = {best_tol[0]:.3f}, overshoot {best_tol[1]:.2f}%, "
      f"settles {best_tol[2]:.3f} s")
print(f"    Note c = {best_tol[0]:.1f} is NOT 'no overshoot' -- it overshoots by "
      f"{best_tol[1]:.2f}%. It is")
print(f"    simply the fastest option once you agree to tolerate a little.")
assert best_zero[1] <= ZERO and best_tol[1] > ZERO, "the two criteria must differ here"
assert best_tol[2] < best_zero[2], "tolerating overshoot should buy speed"

print(f"\n  THE CHOICE FOLLOWS FROM THE REQUIREMENT, and the requirement is the designer's:")
print(f"    - if strictly ZERO overshoot is required, critical damping is the natural")
print(f"      choice: c = c_crit = {c_crit:.3f} N*m*s/rad (zeta = 1) is the fastest")
print(f"      non-oscillatory response for this second-order model, settling in "
      f"{metrics(*simulate(c_crit))[1]:.3f} s.")
print(f"    - if up to {TOL}% overshoot is acceptable, a slightly underdamped design such as")
print(f"      c = {best_tol[0]:.1f} N*m*s/rad settles faster ({best_tol[2]:.3f} s vs "
      f"{best_zero[2]:.3f} s).")
print(f"  Neither is 'the right answer' in the abstract -- small overshoot is perfectly")
print(f"  acceptable in many servo applications, and forbidden in others. What the physics")
print(f"  supplies is the CONSEQUENCE of each requirement, not the requirement itself.")
print(f"\n  BACK TO WHAT P10 ASKS. The required finding is that the as-built joint is")
print(f"  strongly UNDERDAMPED, zeta = {zeta:.3f} < 1, with omega_0 = {w0:.4f} rad/s")
print(f"  and omega_d = {wd:.4f} rad/s. The extension quantifies what that behaviour")
print(f"  looks like: c = {c:.1f} is only {100*c/c_crit:.0f}% of critical, giving "
      f"{metrics(*simulate(c))[0]:.0f}% overshoot")
print(f"  and {metrics(*simulate(c))[1]:.2f} s to settle -- but those numbers are ours, "
      "not the question's.")

# --- Plot ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.8, 4.2))
for cv, col, lbl in ((2.0, "#1565c0", f"c = 2.0 (as built, $\\zeta$={2.0/c_crit:.2f})"),
                     (c_crit, "#2e7d32", f"c = {c_crit:.2f} (critical)"),
                     (20.0, "#e65100", f"c = 20.0 (overdamped)")):
    tt, thh = simulate(cv)
    ax.plot(tt, np.degrees(thh), lw=2, color=col, label=lbl)
ax.axhline(0, c="k", lw=.8)
ax.axhspan(-0.02*10, 0.02*10, color="grey", alpha=.25, label="2% settling band")
ax.set_xlabel("t (s)"); ax.set_ylabel("$\\theta$ (deg)"); ax.set_xlim(0, 2.0)
ax.set_title("W11 P10 — choosing the joint damping")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(w0 - 11.1803) < 1e-3 and abs(gamma - 2.5) < 1e-12
assert abs(zeta - 0.2236) < 1e-3 and zeta < 1
assert abs(c_crit - 8.9443) < 1e-3
print(f"\n[OK] omega_0 = {w0:.2f} rad/s, zeta = {zeta:.3f} (underdamped); "
      f"critical damping would need c = {c_crit:.2f} N*m*s/rad.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_11.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
